# Arabic Text Simplification (AraT5v2)

## Configuration

In [1]:
CSV_PATH = "/kaggle/input/datasets/rayaabualjamal/combined-parallel-dataset/full_merged_data (2).csv"

# Column names 
COL_ORIGINAL = "Original_Sentence"
COL_LEVEL3   = "level_3"   # mild simplification
COL_LEVEL2   = "level_2"   # medium simplification
COL_LEVEL1   = "level_1"   # strong simplification

# Train/dev/test split ratios 
TRAIN_RATIO = 0.80
DEV_RATIO   = 0.10
TEST_RATIO  = 0.10

# ── Output ────────────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/AraT5v2-simplify"

SAVE_TOTAL_LIMIT = 2
SAVE_STEPS = None

# ── Model ─────────────────────────────────────────────────────
MODEL_NAME = "UBC-NLP/AraT5v2-base-1024"

# ── Training pairs to include ─────────────────────────────────
INCLUDE_ORIG_TO_L3  = True   # Original → level_3 (mild)
INCLUDE_ORIG_TO_L2  = True   # Original → level_2 (medium)
INCLUDE_ORIG_TO_L1  = True   # Original → level_1 (strong)
INCLUDE_L3_TO_L2    = False   # level_3  → level_2 (chained)
INCLUDE_L2_TO_L1    = False   # level_2  → level_1 (chained)

# ── Hyperparameters ───────────────────────────────────────────
NUM_EPOCHS              = 15
BATCH_SIZE              = 16
GRADIENT_ACCUMULATION   = 2
LEARNING_RATE           = 3e-5
WARMUP_STEPS            = 500
WEIGHT_DECAY            = 0.01
LABEL_SMOOTHING         = 0.1
MAX_SOURCE_LENGTH       = 128
MAX_TARGET_LENGTH       = 128
EARLY_STOPPING_PATIENCE = 3

# ── Generation ────────────────────────────────────────────────
NUM_BEAMS            = 4
NO_REPEAT_NGRAM_SIZE = 3
LENGTH_PENALTY       = 1.0
MIN_GEN_LENGTH       = 10

# ── Evaluation ────────────────────────────────────────────────
RUN_BERTSCORE   = True
BERTSCORE_MODEL = "bert-base-multilingual-cased"

SEED = 42

# ── Debug / fast-test mode ────────────────────────────────────
DEBUG_MODE       = False 
DEBUG_TRAIN_ROWS = 200
DEBUG_DEV_ROWS   = 50
DEBUG_EPOCHS     = 2

print("Configuration loaded")
print(f"  CSV path  : {CSV_PATH}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Model     : {MODEL_NAME}")
print(f"  Pairs     : Orig→L3={INCLUDE_ORIG_TO_L3}, Orig→L2={INCLUDE_ORIG_TO_L2}, "
      f"Orig→L1={INCLUDE_ORIG_TO_L1}, L3→L2={INCLUDE_L3_TO_L2}, L2→L1={INCLUDE_L2_TO_L1}")

Configuration loaded
  CSV path  : /kaggle/input/datasets/rayaabualjamal/combined-parallel-dataset/full_merged_data (2).csv
  Output dir: /kaggle/working/AraT5v2-simplify
  Model     : UBC-NLP/AraT5v2-base-1024
  Pairs     : Orig→L3=True, Orig→L2=True, Orig→L1=True, L3→L2=False, L2→L1=False


## Install Dependencies

In [2]:
import subprocess, sys

def pip(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Upgrade pip
pip(["--upgrade", "pip"])

# Uninstall potentially incompatible packages
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
                       "transformers", "accelerate", "peft"])

# ── Reinstall PyTorch with the correct CUDA 12.1 binaries ──
pip([
    "torch==2.3.0",
    "torchvision==0.18.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])

# Install the rest pinned
pip([
    "transformers==4.41.2",
    "accelerate==0.30.1",
    "peft==0.11.1",
    "sentencepiece",
    "sacrebleu",
    "bert-score",
    "git+https://github.com/feralvam/easse.git",
])

print("All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.9 MB/s eta 0:00:00
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, which is not installed.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.0+cu121 which is incompatible.


All packages installed


## Imports & Seed

In [3]:
# ── Install packages ──────────────────────────────────────────
import subprocess, sys

# BARTScore 
subprocess.run([
    "wget", "-q",
    "https://raw.githubusercontent.com/neulab/BARTScore/main/bart_score.py",
    "-O", "/kaggle/working/bart_score.py"
])

# camel-tools without touching numpy
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "camel-tools", "--no-deps", "-q"
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "camel-kenlm", "pyrsistent", "muddler", "docopt", "cachetools", "-q"
])

# Fix numpy back to 2.x after camel-tools install
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "numpy>=2.0", "-q"
])

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("All packages ready")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
camel-tools 1.5.7 requires cachetools<=6.0.0, but you have cachetools 6.2.6 which is incompatible.
camel-tools 1.5.7 requires camel-kenlm<=2025.09.16; platform_system != "Windows", but you have camel-kenlm 2026.2.7 which is incompatible.
camel-tools 1.5.7 requires numpy<2, but you have numpy 2.0.2 which is incompatible.


All packages ready


In [4]:
import sys
sys.path.insert(0, "/kaggle/working")  # so Python finds bart_score.py
from bart_score import BARTScorer

In [5]:
import os, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, EarlyStoppingCallback,
    set_seed,
)
import sacrebleu
from easse.sari import corpus_sari
import math
from camel_tools.utils.normalize import normalize_unicode
from camel_tools.sentiment import SentimentAnalyzer 

warnings.filterwarnings("ignore")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Imports complete")

2026-04-21 17:42:42.813913: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776793363.002659      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776793363.051622      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776793363.464661      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776793363.464709      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776793363.464713      23 computation_placer.cc:177] computation placer alr

Device: cuda
  GPU: Tesla P100-PCIE-16GB
  VRAM: 17.1 GB
Imports complete


## Load & Validate CSV

In [6]:
# ── Load CSV ──────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Loaded CSV: {len(df):,} rows, columns: {list(df.columns)}")

# ── Validate required columns ─────────────────────────────────
required = [COL_ORIGINAL, COL_LEVEL3, COL_LEVEL2, COL_LEVEL1]
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}\nFound: {list(df.columns)}")

# ── Basic cleaning ────────────────────────────────────────────
df = df[required].copy()
df = df.dropna(subset=required)           # drop rows with any NaN in required cols
df = df[df[COL_ORIGINAL].str.strip() != ""]  # drop empty original sentences
df = df.reset_index(drop=True)

print(f"After cleaning: {len(df):,} rows")
print("\nSample rows:")
print(df.head(3).to_string())

Loaded CSV: 34,294 rows, columns: ['Original_Sentence', 'level_1', 'level_2', 'level_3', 'evaluation_status', 'issue_details']
After cleaning: 34,294 rows

Sample rows:
                                                                                                                                  Original_Sentence                                                                                                                                           level_3                                                                                                            level_2                                                            level_1
0                                                                                                      •• "عدم الانحياز" اصطلاح سياسي يعنى الحياد..                                                                                                    "عدم الانحياز" هو مصطلح سياسي يشير إلى الحياد.                                                              

## Train / Dev / Test Split

In [7]:
# Split indices
train_df, temp_df = train_test_split(
    df, test_size=(DEV_RATIO + TEST_RATIO), random_state=SEED
)
dev_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_RATIO / (DEV_RATIO + TEST_RATIO),
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
dev_df   = dev_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,}  |  Dev: {len(dev_df):,}  |  Test: {len(test_df):,}")

# Debug mode: shrink sets
if DEBUG_MODE:
    train_df = train_df.head(DEBUG_TRAIN_ROWS)
    dev_df   = dev_df.head(DEBUG_DEV_ROWS)
    test_df  = test_df.head(DEBUG_DEV_ROWS)
    print(f"[DEBUG] Train: {len(train_df)}  Dev: {len(dev_df)}  Test: {len(test_df)}")

Train: 27,435  |  Dev: 3,429  |  Test: 3,430


## Build Training Pairs

In [8]:
# ── Level tokens (control tags prepended to source) ───────────
# [L3] = mild, [L2] = medium, [L1] = strong
TOKEN_MAP = {
    "orig_to_l3" : "[L3]",
    "orig_to_l2" : "[L2]",
    "orig_to_l1" : "[L1]",
    "l3_to_l2"   : "[L2]",
    "l2_to_l1"   : "[L1]",
}

def make_pairs(df_split, include_flags):
    """Build a list of (source_with_tag, target) tuples for all active pairs."""
    rows = []

    if include_flags.get("orig_to_l3"):
        for _, r in df_split.iterrows():
            rows.append({"source": f"[L3] {r[COL_ORIGINAL]}", "target": str(r[COL_LEVEL3])})

    if include_flags.get("orig_to_l2"):
        for _, r in df_split.iterrows():
            rows.append({"source": f"[L2] {r[COL_ORIGINAL]}", "target": str(r[COL_LEVEL2])})

    if include_flags.get("orig_to_l1"):
        for _, r in df_split.iterrows():
            rows.append({"source": f"[L1] {r[COL_ORIGINAL]}", "target": str(r[COL_LEVEL1])})

    if include_flags.get("l3_to_l2"):
        for _, r in df_split.iterrows():
            rows.append({"source": f"[L2] {r[COL_LEVEL3]}", "target": str(r[COL_LEVEL2])})

    if include_flags.get("l2_to_l1"):
        for _, r in df_split.iterrows():
            rows.append({"source": f"[L1] {r[COL_LEVEL2]}", "target": str(r[COL_LEVEL1])})

    return pd.DataFrame(rows)

INCLUDE_FLAGS = {
    "orig_to_l3" : INCLUDE_ORIG_TO_L3,
    "orig_to_l2" : INCLUDE_ORIG_TO_L2,
    "orig_to_l1" : INCLUDE_ORIG_TO_L1,
    "l3_to_l2"   : INCLUDE_L3_TO_L2,
    "l2_to_l1"   : INCLUDE_L2_TO_L1,
}

train_pairs = make_pairs(train_df, INCLUDE_FLAGS)
dev_pairs   = make_pairs(dev_df,   INCLUDE_FLAGS)
test_pairs  = make_pairs(test_df,  INCLUDE_FLAGS)

print(f"Training pairs  : {len(train_pairs):,}")
print(f"Dev pairs       : {len(dev_pairs):,}")
print(f"Test pairs      : {len(test_pairs):,}")
print("\nSample training pairs:")
print(train_pairs.head(6).to_string())

Training pairs  : 82,305
Dev pairs       : 10,287
Test pairs      : 10,290

Sample training pairs:
                                                                                                                                                                                              source                                                                                                                                               target
0                                                                                                                                                 [L3] التي يرسلونها على الجيوش كأنها قطع من الجحيم،                                                                                                             التي يرسلونها إلى الجيوش كقطع من الجحيم.
1                                        [L3] تستخدم فيها الألوان والظلال لتمثيل ظاهرات تنتشر على مساحات معينة كانتشار الغابات.  (أ) رموز خطية  (ب) رموز نقطية  (ج) رموز مساحية  (د) نظام الإحداثيات             

## Tokenizer & Dataset

In [9]:
# ── Load tokenizer and add level tokens ───────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

NEW_TOKENS = ["[L1]", "[L2]", "[L3]"]
added = tokenizer.add_tokens(NEW_TOKENS)
print(f"Added {added} new control tokens: {NEW_TOKENS}")

# ── Build HuggingFace DatasetDict ─────────────────────────────
def df_to_hf(df_pairs):
    return Dataset.from_dict({
        "source": df_pairs["source"].tolist(),
        "target": df_pairs["target"].tolist(),
    })

raw_datasets = DatasetDict({
    "train" : df_to_hf(train_pairs),
    "dev"   : df_to_hf(dev_pairs),
    "test"  : df_to_hf(test_pairs),
})
print(raw_datasets)

# ── Tokenization function ─────────────────────────────────────
def preprocess(batch):
    model_inputs = tokenizer(
        batch["source"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False,
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["target"],
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
            padding=False,
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = raw_datasets.map(
    preprocess, batched=True,
    remove_columns=["source", "target"],
    desc="Tokenizing",
)
print("Tokenization complete ✓")
print(tokenized)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/2.35M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Added 3 new control tokens: ['[L1]', '[L2]', '[L3]']
DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 82305
    })
    dev: Dataset({
        features: ['source', 'target'],
        num_rows: 10287
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 10290
    })
})


Tokenizing:   0%|          | 0/82305 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/10287 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/10290 [00:00<?, ? examples/s]

Tokenization complete ✓
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 82305
    })
    dev: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10287
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10290
    })
})


## Load Model & Train

In [10]:
import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free disk space: {free_gb:.1f} GB")
if free_gb < 5:
    print("WARNING: Less than 5GB free — consider reducing NUM_EPOCHS or SAVE_TOTAL_LIMIT")
else:
    print("Disk space looks fine")

Free disk space: 20.9 GB
Disk space looks fine


In [11]:
import math

# ── Load model ────────────────────────────────────────────────
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))  # accommodate new [L1/L2/L3] tokens
print(f"Model loaded ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

# ── Data collator ─────────────────────────────────────────────
collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)

# ── Training arguments ────────────────────────────────────────
actual_epochs = DEBUG_EPOCHS if DEBUG_MODE else NUM_EPOCHS

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=actual_epochs,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    label_smoothing_factor=LABEL_SMOOTHING,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=NUM_BEAMS,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
)

# ── Trainer ───────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["dev"],
    tokenizer=tokenizer,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("Starting training...")
trainer.train()
print("Training complete")

# ── Save final model ──────────────────────────────────────────
final_model_dir = os.path.join(OUTPUT_DIR, "model-joint")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"Model saved → {final_model_dir}")

config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Model loaded (367M params)
Starting training...


Epoch,Training Loss,Validation Loss
0,3.759700,3.271945
2,3.397200,3.096163
4,3.167500,3.038723
6,3.089000,3.026973
8,2.985500,3.007038
10,2.918600,3.014075
12,2.923500,3.011501


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Training complete
Model saved → /kaggle/working/AraT5v2-simplify/model-joint


## Evaluation (SARI + BLEU + BERTScore + BARTScore + OSMAN)

In [12]:
from bert_score import score as bert_score_fn

def generate_predictions(model, tokenizer, sources, level_token,
                         batch_size=8, min_length=10):
    """Generate simplified sentences for a list of source strings."""
    model.eval()
    preds = []
    for i in range(0, len(sources), batch_size):
        batch_src = [f"{level_token} {s}" for s in sources[i:i+batch_size]]
        enc = tokenizer(
            batch_src, return_tensors="pt",
            padding=True, truncation=True,
            max_length=MAX_SOURCE_LENGTH,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                num_beams=NUM_BEAMS,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
                length_penalty=LENGTH_PENALTY,
                min_length=min_length,
                max_length=MAX_TARGET_LENGTH,
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        preds.extend(decoded)
        print(f"Generated {min(i+batch_size, len(sources))}/{len(sources)}", end="\r")
    print()
    return preds


print("Loading BARTScorer...")
bart_scorer = BARTScorer(device=DEVICE, checkpoint="facebook/bart-large-cnn")
print("BARTScorer loaded")

# ── OSMAN helper ──────────────────────────────────────────────
def osman_score(hypothesis, reference):
    """
    OSMAN: Arabic MT metric based on modified BLEU + penalty for broken words.
    Approximation using sacrebleu with Arabic normalization + unigram F1.
    For full OSMAN, see: https://github.com/nltk/nltk (nltk.translate.meteor_score)
    """
    from nltk.translate.meteor_score import meteor_score
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)

    scores = []
    for hyp, ref in zip(hypothesis, reference):
        # Normalize Arabic text
        hyp_norm = normalize_unicode(hyp)
        ref_norm = normalize_unicode(ref)
        # Tokenize
        hyp_tokens = hyp_norm.split()
        ref_tokens = ref_norm.split()
        if not hyp_tokens or not ref_tokens:
            scores.append(0.0)
            continue
        score = meteor_score([ref_tokens], hyp_tokens)
        scores.append(score)
    return float(np.mean(scores)) * 100  # return as percentage like BLEU


def evaluate_direction(model, tokenizer, sources, references, level_token, label):
    print(f"\n{'='*60}")
    print(f"Evaluating: {label}")
    print(f"{'='*60}")

    preds = generate_predictions(model, tokenizer, sources, level_token)

    # ── SARI ──────────────────────────────────────────────────
    sari = corpus_sari(
        orig_sents=sources,
        sys_sents=preds,
        refs_sents=[references],
    )
    print(f"SARI        : {sari:.2f}")

    # ── BLEU ──────────────────────────────────────────────────
    bleu_score = sacrebleu.corpus_bleu(preds, [references]).score
    print(f"BLEU        : {bleu_score:.2f}")

    result = {"direction": label, "sari": sari, "bleu": bleu_score}

    # ── BERTScore ─────────────────────────────────────────────
    if RUN_BERTSCORE:
        P, R, F1 = bert_score_fn(
            preds, references,
            lang="ar",
            verbose=False,
        )
        bs = F1.mean().item()
        print(f"BERTScore F1: {bs:.4f}")
        result["bertscore_f1"] = bs

    # ── BARTScore ─────────────────────────────────────────────
    bart_scores = bart_scorer.score(
        preds, references,
        batch_size=8,
    )
    bart_avg = float(np.mean(bart_scores))
    print(f"BARTScore   : {bart_avg:.4f}")
    result["bartscore"] = bart_avg

    # ── OSMAN ─────────────────────────────────────────────────
    osman = osman_score(preds, references)
    print(f"OSMAN       : {osman:.2f}")
    result["osman"] = osman

    return result, preds


# ── Run evaluation on test split ──────────────────────────────
all_results = []

if INCLUDE_ORIG_TO_L3:
    res, _ = evaluate_direction(
        trainer.model, tokenizer,
        test_df[COL_ORIGINAL].tolist(),
        test_df[COL_LEVEL3].tolist(),
        "[L3]", "Original → level_3",
    )
    all_results.append(res)

if INCLUDE_ORIG_TO_L2:
    res, _ = evaluate_direction(
        trainer.model, tokenizer,
        test_df[COL_ORIGINAL].tolist(),
        test_df[COL_LEVEL2].tolist(),
        "[L2]", "Original → level_2",
    )
    all_results.append(res)

if INCLUDE_ORIG_TO_L1:
    res, _ = evaluate_direction(
        trainer.model, tokenizer,
        test_df[COL_ORIGINAL].tolist(),
        test_df[COL_LEVEL1].tolist(),
        "[L1]", "Original → level_1",
    )
    all_results.append(res)

if INCLUDE_L3_TO_L2:
    res, _ = evaluate_direction(
        trainer.model, tokenizer,
        test_df[COL_LEVEL3].tolist(),
        test_df[COL_LEVEL2].tolist(),
        "[L2]", "level_3 → level_2 (chained)",
    )
    all_results.append(res)

if INCLUDE_L2_TO_L1:
    res, _ = evaluate_direction(
        trainer.model, tokenizer,
        test_df[COL_LEVEL2].tolist(),
        test_df[COL_LEVEL1].tolist(),
        "[L1]", "level_2 → level_1 (chained)",
    )
    all_results.append(res)

# ── Save results ──────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_path = os.path.join(OUTPUT_DIR, "final_results.csv")
results_df.to_csv(results_path, index=False)
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60) 
print(results_df.to_string(index=False))

Loading BARTScorer...


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore F1: 0.8701
BARTScore   : -3.1535
OSMAN       : 50.36

Evaluating: Original → level_2
Generated 3430/3430
SARI        : 46.13
BLEU        : 26.26
BERTScore F1: 0.8547
BARTScore   : -3.2627
OSMAN       : 44.63

Evaluating: Original → level_1
Generated 3430/3430
SARI        : 43.84
BLEU        : 14.79
BERTScore F1: 0.8179
BARTScore   : -3.4997
OSMAN       : 31.36

FINAL RESULTS
         direction      sari      bleu  bertscore_f1  bartscore     osman
Original → level_3 45.962188 31.134753      0.870100  -3.153526 50.364988
Original → level_2 46.133812 26.257645      0.854738  -3.262690 44.631676
Original → level_1 43.838789 14.789725      0.817903  -3.499733 31.361635


## Demo: Interactive Simplification

In [13]:
sample_texts = [

    # Literary / narrative prose (SAMER's primary domain)
    "فلما أسدل الليل ستائره على المدينة النائمة، خرج الفتى من مخبئه متسللاً بين الظلال.",
    "وكان الشيخ يجلس في زاوية المسجد مستغرقاً في تلاوة آيات الذكر الحكيم بصوت خافت.",
    "اضطرمت في صدره نيران الغيرة حين رأى منافسه يحصد ثمار جهوده دون أن يبذل جهداً يُذكر.",

    # Complex subordinate clauses
    "أدرك الرجل المتعب الذي قضى سنوات طويلة في الغربة بعيداً عن أهله أن العودة إلى الوطن ليست سهلة.",
    "وبينما كانت الأم تُهدهد طفلها الصغير الباكي، دخل الأب حاملاً بشرى الفرج بعد طول انتظار.",

    # Formal/archaic vocabulary
    "سعى الوزير الحكيم إلى استتباب الأمن وإرساء دعائم العدل في ربوع المملكة المترامية الأطراف.",
    "إن الإنسان الذي لا يتعظ بتجارب الماضي ولا يستفيد من دروس التاريخ محكوم عليه بتكرار أخطائه.",

    # Dense nominal phrases (hard to simplify)
    "تجلت عبقرية الشاعر في توظيفه البديع للصور الاستعارية المعبرة عن وجدانه المضطرب.",
    "استأثرت قضية الإصلاح الاقتصادي باهتمام المفكرين والباحثين في شتى أرجاء العالم العربي.",

    # Long sentences with multiple clauses (stress test)
    "حين أقبل الربيع بنسائمه العليلة وأزهاره المتفتحة، شعر العجوز الذي أمضى شتاءً قاسياً وحيداً بأن الحياة لا تزال تستحق أن تُعاش.",
]

print("=" * 70)
print("DEMO — Level-Conditioned Simplification")
print("=" * 70)

for text in sample_texts:
    print(f"\nOriginal:\n  {text}")
    for token, label in [("[L3]", "level_3 (mild)"), ("[L2]", "level_2 (medium)"), ("[L1]", "level_1 (strong)")]:
        pred = generate_predictions(
            trainer.model, tokenizer,
            [text], token, batch_size=1, min_length=MIN_GEN_LENGTH
        )[0]
        print(f"{label}:\n  {pred}")
    print("-" * 70)

DEMO — Level-Conditioned Simplification

Original:
  فلما أسدل الليل ستائره على المدينة النائمة، خرج الفتى من مخبئه متسللاً بين الظلال.
Generated 1/1
level_3 (mild):
  خرج الفتى من مخبئه متسللاً بين الظلال.
Generated 1/1
level_2 (medium):
  خرج الفتى من مخبئه متسللاً بين الظلال.
Generated 1/1
level_1 (strong):
  خرج الفتى من مخبئه متسللاً بين الظلال.
----------------------------------------------------------------------

Original:
  وكان الشيخ يجلس في زاوية المسجد مستغرقاً في تلاوة آيات الذكر الحكيم بصوت خافت.
Generated 1/1
level_3 (mild):
  كان الشيخ يجلس في زاوية المسجد، يقرأ آيات الذكر الحكيم بصوت خافت.
Generated 1/1
level_2 (medium):
  كان الشيخ يجلس في زاوية المسجد، يقرأ آيات الذكر الحكيم بصوت خافت.
Generated 1/1
level_1 (strong):
  كان الشيخ يجلس في زاوية المسجد، يقرأ آيات القرآن بصوت خافت.
----------------------------------------------------------------------

Original:
  اضطرمت في صدره نيران الغيرة حين رأى منافسه يحصد ثمار جهوده دون أن يبذل جهداً يُذكر.
Generated 1/1
level_3 (m

## List Output Files

In [14]:
import os
print("Files in output directory:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs[:] = [d for d in dirs if not d.startswith("checkpoint") and d not in ("logs", "__pycache__")]
    lvl = root.replace(OUTPUT_DIR, "").count(os.sep)
    print("  " * lvl + os.path.basename(root) + "/")
    for f in sorted(files):
        sz = os.path.getsize(os.path.join(root, f))
        s  = f"{sz/1e6:.1f}MB" if sz > 1e6 else f"{sz/1e3:.0f}KB"
        print("  " * (lvl + 1) + f"{f}  ({s})")
print("\nDownload via: Notebook > Data tab > Output > Download as zip")

Files in output directory:
AraT5v2-simplify/
  final_results.csv  (0KB)
  model-joint/
    added_tokens.json  (0KB)
    config.json  (1KB)
    generation_config.json  (0KB)
    model.safetensors  (1469.4MB)
    special_tokens_map.json  (3KB)
    spiece.model  (2.4MB)
    tokenizer.json  (8.4MB)
    tokenizer_config.json  (22KB)
    training_args.bin  (5KB)

Download via: Notebook > Data tab > Output > Download as zip
